# 00. 데이터 수집·정제

ABS 도입 이전 두 시즌(2024-2025)의 MLB 정규시즌 투구 데이터를 받아,
**심판이 볼/스트라이크를 직접 판정한 공**만 남기고 분석에 필요한 변수를
붙여 `data/clean.parquet`으로 저장한다. 이후 모든 노트북은 이 파일 하나에서
출발한다.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path("../data")
MODEL_DIR = Path("../models")

PLATE_HALF = 0.708            # 홈플레이트 반폭 (ft)
BALL_RADIUS = 0.121           # 공 반지름 (ft)
BORDERLINE_THRESHOLD = 0.121  # 경계 공 기준 (ft)
MIN_CELL_N = 200              # 셀당 최소 표본 수
SPLIT_DATE = "2025-08-01"     # train/test 분리 날짜 (4.3절)

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

이 노트북에서만 쓰는 상수다.

- **시즌 기간**은 각 시즌 개막일부터 정규시즌 최종일까지다. 2024년은 서울
  시리즈(3월 20일), 2025년은 도쿄 시리즈(3월 18일)로 개막했다. 포스트시즌은
  이 기간 밖이고, 혹시 섞여 들어와도 `game_type` 필터에서 빠진다.
- **판정구**는 타자가 스윙하지 않아 심판이 존 안팎을 판단한 공이다.
  `blocked_ball`은 원바운드 등으로 포수가 막아낸 볼이며 역시 볼 판정이다.
  `automatic_ball`/`automatic_strike`는 피치 클락 위반 등으로 투구 위치와
  무관하게 주어지는 판정이라 넣지 않는다.
- **`KEEP_COLS`**는 원본 119개 컬럼 중 실제로 쓰는 30개다 (가이드라인 1.3절).

In [2]:
import contextlib
import io
import urllib.request
import warnings

from pybaseball import statcast

RAW_DIR = DATA_DIR / "raw"

SEASONS = {
    2024: ("2024-03-20", "2024-09-30"),
    2025: ("2025-03-18", "2025-09-28"),
}

CALLED = ["called_strike", "ball", "blocked_ball"]
SZ_TOP_MAX = 5.0  # 존 상단이 이보다 높으면 측정 오류로 본다 (ft)

KEEP_COLS = [
    "game_date", "game_pk", "game_year", "game_type",
    "at_bat_number", "pitch_number",
    "description",
    "plate_x", "plate_z", "sz_top", "sz_bot",
    "balls", "strikes",
    "outs_when_up", "inning", "inning_topbot",
    "on_1b", "on_2b", "on_3b",
    "bat_score", "fld_score",
    "pitcher", "batter", "fielder_2",
    "stand", "p_throws",
    "pitch_type", "release_speed",
    "home_team", "away_team",
]

FRAMING_URL = (
    "https://baseballsavant.mlb.com/leaderboard/catcher-framing"
    "?type=catcher&seasonStart={year}&seasonEnd={year}&team=&min=q"
    "&sortColumn=rv_tot&sortDirection=desc&csv=true"
)

## 1. 수집

데이터는 MLB 공식 투구 추적 시스템 [Statcast](https://baseballsavant.mlb.com/statcast_search)를
[pybaseball](https://github.com/jldbc/pybaseball)로 받는다. 투구마다 홈플레이트를
통과한 위치(`plate_x`, `plate_z`), 타자별 스트라이크 존 높이(`sz_top`, `sz_bot`),
심판 판정(`description`)이 기록돼 있다.

두 가지를 지킨다.

- **월 단위로 나눠 받는다.** 긴 기간을 한 번에 요청하면 오래 걸리고, 중간에
  끊기거나 행 수 상한에 걸려 조용히 잘릴 수 있다.
- **받은 즉시 원본을 `data/raw/`에 저장하고, 이미 있는 달은 건너뛴다.**
  재실행해도 다시 받지 않고, 끊겨도 이어서 받는다. 원본은 용량이 커서
  git에 올리지 않는다 (가이드라인 5.4절).

In [3]:
def month_ranges(start, end):
    """[start, end] 기간을 달력 월 단위 (시작일, 종료일) 목록으로 자른다."""
    cur, end = pd.Timestamp(start), pd.Timestamp(end)
    ranges = []
    while cur <= end:
        month_end = min(cur + pd.offsets.MonthEnd(0), end)
        ranges.append((cur, month_end))
        cur = month_end + pd.Timedelta(days=1)
    return ranges


def fetch_month(start, end):
    """한 달 치 원본을 받아 저장한다. 이미 있으면 받지 않는다."""
    path = RAW_DIR / f"statcast_{start:%Y_%m}.parquet"
    if path.exists():
        return path, "기존 파일"
    # pybaseball의 진행 막대와 경고는 노트북 출력만 어지럽혀 숨긴다
    with warnings.catch_warnings(), contextlib.redirect_stderr(io.StringIO()):
        warnings.simplefilter("ignore")
        df = statcast(start_dt=f"{start:%Y-%m-%d}", end_dt=f"{end:%Y-%m-%d}", verbose=False)
    df.to_parquet(path, index=False)
    return path, "새로 받음"


RAW_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for year, (start, end) in SEASONS.items():
    for s, e in month_ranges(start, end):
        path, status = fetch_month(s, e)
        ids = pd.read_parquet(path, columns=["game_pk"])["game_pk"]
        rows.append({
            "시즌": year,
            "파일": path.name,
            "기간": f"{s:%m-%d} ~ {e:%m-%d}",
            "투구 수": len(ids),
            "경기 수": ids.nunique(),
            "경기당 투구": round(len(ids) / ids.nunique()),
            "상태": status,
        })

raw_summary = pd.DataFrame(rows)
raw_summary

,시즌,파일,기간,투구 수,경기 수,경기당 투구,상태
0,2024,statcast_2024_03.parquet,03-20 ~ 03-31,37217,143,260,기존 파일
1,2024,statcast_2024_04.parquet,04-01 ~ 04-30,116769,397,294,기존 파일
2,2024,statcast_2024_05.parquet,05-01 ~ 05-31,118739,409,290,기존 파일
3,2024,statcast_2024_06.parquet,06-01 ~ 06-30,116355,401,290,기존 파일
4,2024,statcast_2024_07.parquet,07-01 ~ 07-31,108693,369,295,기존 파일
5,2024,statcast_2024_08.parquet,08-01 ~ 08-31,121441,413,294,기존 파일
6,2024,statcast_2024_09.parquet,09-01 ~ 09-30,113268,385,294,기존 파일
7,2025,statcast_2025_03.parquet,03-18 ~ 03-31,49338,168,294,기존 파일
8,2025,statcast_2025_04.parquet,04-01 ~ 04-30,114767,391,294,기존 파일
9,2025,statcast_2025_05.parquet,05-01 ~ 05-31,120218,411,293,기존 파일


월별 투구 수는 10만~12만 구, 경기당 290구 안팎으로 고르다. **유독 적은 달은 없다.** 3월은 개막 후 며칠뿐이라 적고, 3월 파일에는 시범경기도 섞여 있다 (다음 절의 정규시즌 필터에서 빠진다). 7월은 올스타 휴식기가 있어 경기 수가 조금 적다.

In [4]:
raw_summary.groupby("시즌")[["투구 수", "경기 수"]].sum()

,투구 수,경기 수
시즌,,
2024,732482,2517
2025,742080,2531


두 시즌 합계 약 147만 구다. 시범경기가 섞인 수치이며, 정규시즌만 남긴 수치는 아래 정제 표에 있다.

원본에는 `umpire` 컬럼도 있다. 심판별 분석이 가능하다면 큰 도움이 되므로
실제로 값이 들어 있는지 확인한다.

In [5]:
raw_files = sorted(RAW_DIR.glob("statcast_*.parquet"))
umpire = pd.concat(pd.read_parquet(p, columns=["umpire"]) for p in raw_files)["umpire"]
print(f"umpire 값이 있는 행: {umpire.notna().sum():,} / {len(umpire):,}")

umpire 값이 있는 행: 0 / 1,474,562


**모든 행이 비어 있다.** Statcast는 심판 정보를 제공하지 않는다. 따라서 심판 개인별 분석은 하지 않고, 모든 판정을 리그 심판 전체의 판정으로 본다.

## 2. 필요한 컬럼만 불러오기

원본 119개 컬럼 중 `KEEP_COLS` 30개만 읽는다. 용량을 줄이고, 어떤 컬럼을
왜 쓰는지 분명히 하기 위해서다. 각 컬럼의 뜻은
[Baseball Savant CSV 문서](https://baseballsavant.mlb.com/csv-docs)에 있다.

In [6]:
raw = pd.concat(
    (pd.read_parquet(p, columns=KEEP_COLS) for p in raw_files),
    ignore_index=True,
)
print(f"{len(raw):,}행 × {raw.shape[1]}열")
raw.head()

1,474,562행 × 30열


,game_date,game_pk,game_year,game_type,at_bat_number,pitch_number,description,plate_x,plate_z,sz_top,...,fld_score,pitcher,batter,fielder_2,stand,p_throws,pitch_type,release_speed,home_team,away_team
0,2024-03-31,747222,2024,R,66,7,swinging_strike,0.077266,3.227396,3.42,...,5,656464,686668,571466,R,R,FF,95.7,AZ,COL
1,2024-03-31,747222,2024,R,66,6,foul,-0.044209,1.73542,3.42,...,5,656464,686668,571466,R,R,SI,94.6,AZ,COL
2,2024-03-31,747222,2024,R,66,5,blocked_ball,1.722746,0.268207,3.41638,...,5,656464,686668,571466,R,R,SL,88.4,AZ,COL
3,2024-03-31,747222,2024,R,66,4,foul,0.270552,1.738154,3.42,...,5,656464,686668,571466,R,R,SL,86.9,AZ,COL
4,2024-03-31,747222,2024,R,66,3,ball,0.162886,1.022717,3.339202,...,5,656464,686668,571466,R,R,SI,94.7,AZ,COL


## 3. 정제

가이드라인 1.4절 순서대로 거른다. **단계마다 남은 행과 빠진 행을 기록한다.**
어떤 공이 왜 빠졌는지가 재현성의 핵심이다.

In [7]:
clean_log = []


def record(step, df):
    prev = clean_log[-1]["남은 행"] if clean_log else len(df)
    clean_log.append({"단계": step, "남은 행": len(df), "제거된 행": prev - len(df)})


record("0. 원본 (2024-2025 전체 투구)", raw)

regular = raw[raw["game_type"] == "R"]
record("1. 정규시즌만", regular)

regular = regular.drop_duplicates(["game_pk", "at_bat_number", "pitch_number"])
record("1-1. 중복 투구 제거", regular)

### 선발 투수 표시 (`is_starter`)

선수 축 분석에서는 선발과 불펜을 섞지 않는다 (가이드라인 3.2절). 등판
상황이 달라 상대 타자와 카운트 분포가 다르기 때문이다. 경기마다 각 팀의
첫 투구를 던진 투수를 그 경기의 선발로 본다.

**판정구로 거르기 전에 계산해야 한다.** 판정구만 남기면 경기의 첫 투구가
빠질 수 있다 (첫 공에 스윙한 경우).

In [8]:
starters = (
    regular.sort_values(["game_pk", "at_bat_number", "pitch_number"])
    .drop_duplicates(["game_pk", "inning_topbot"])[["game_pk", "pitcher"]]
    .assign(is_starter=1)
)
print(f"선발 기록: {len(starters):,}건 (경기 {regular['game_pk'].nunique():,}개 × 2팀)")

선발 기록: 9,718건 (경기 4,859개 × 2팀)


### 판정구, 결측, 이상치

- **판정구만 남긴다.** 스윙·파울·인플레이·몸에 맞는 공은 심판의 존 판단이 아니다.
- **위치 결측을 뺀다.** 추적 장비가 공을 놓친 경우다.
- **존 이상치를 뺀다.** 존 상단이 하단보다 낮거나, 5ft(약 152cm)를 넘는 경우다.

In [9]:
called = regular[regular["description"].isin(CALLED)]
record("2. 판정구만", called)

called = called.dropna(subset=["plate_x", "plate_z", "sz_top", "sz_bot"])
record("3. 위치 결측 제거", called)

called[["sz_top", "sz_bot"]].describe().round(2)

,sz_top,sz_bot
count,735718.0,735718.0
mean,3.42,1.6
std,0.2,0.12
min,2.51,1.0
25%,3.28,1.52
50%,3.42,1.6
75%,3.55,1.68
max,4.41,2.14


존 상단은 2.5~4.4ft, 하단은 1.0~2.1ft 범위로 모두 현실적인 값이다. 하단이 상단보다 높거나 상단이 5ft를 넘는 값은 없다.

In [10]:
bad_zone = (called["sz_top"] <= called["sz_bot"]) | (called["sz_top"] > SZ_TOP_MAX)
called = called[~bad_zone]
record("4. 존 이상치 제거", called)

pd.DataFrame(clean_log)

,단계,남은 행,제거된 행
0,0. 원본 (2024-2025 전체 투구),1474562,0
1,1. 정규시즌만,1424427,50135
2,1-1. 중복 투구 제거,1424427,0
3,2. 판정구만,735942,688485
4,3. 위치 결측 제거,735718,224
5,4. 존 이상치 제거,735718,0


- **정규시즌 필터**에서 빠진 약 5만 구는 3월 시범경기다.
- **중복 투구는 없었다.**
- 정규시즌 투구의 **51.7%가 판정구**다. 나머지 절반은 스윙·파울·인플레이 등 심판의 존 판단이 아닌 공이다.
- 위치 결측은 224구(판정구의 0.03%)뿐이고, 존 이상치는 없었다.

최종 **735,718구**가 남았다. 정규시즌 4,859경기(2024년 2,429, 2025년 2,430)에서 나온 공이다.

## 4. 파생 변수

가이드라인 2절과 3.1절의 정의를 그대로 쓴다.

**스트라이크 존**은 홈플레이트 폭(17인치)과 타자의 무릎~가슴 높이로 정해지는
가상의 상자다. 규칙상 **공이 존에 조금이라도 걸치면** 스트라이크다.
그래서 존 경계까지의 거리(`zone_dist`)를 잴 때 좌우·상하 모두 공 반지름만큼
존을 넓혀 잡는다. `zone_dist`는 존 안쪽이 양수, 바깥쪽이 음수이며, 좌우와
상하 중 더 가까운 경계까지의 거리다.

`plate_z_norm`은 타자마다 다른 존 높이를 0(하단)~1(상단)로 맞춘 투구 높이다.

In [11]:
clean = called.copy()

for col in ["plate_x", "plate_z", "sz_top", "sz_bot", "release_speed"]:
    clean[col] = clean[col].astype("float64")

clean["is_strike"] = (clean["description"] == "called_strike").astype("int8")
clean["plate_z_norm"] = (clean["plate_z"] - clean["sz_bot"]) / (clean["sz_top"] - clean["sz_bot"])

dx = (PLATE_HALF + BALL_RADIUS) - clean["plate_x"].abs()
dz = np.minimum(
    (clean["sz_top"] + BALL_RADIUS) - clean["plate_z"],
    clean["plate_z"] - (clean["sz_bot"] - BALL_RADIUS),
)
clean["zone_dist"] = np.minimum(dx, dz)
clean["is_borderline"] = (clean["zone_dist"].abs() <= BORDERLINE_THRESHOLD).astype("int8")

상황 변수다. **볼카운트**는 투구 전 볼과 스트라이크 개수(예: 3-0은 볼 3개,
스트라이크 0개)이고 원본 `balls`, `strikes`를 그대로 쓴다.

- `score_diff`: 타석에 선 팀 점수 − 수비 팀 점수
- `risp`: 2루나 3루에 주자가 있음 (득점권)
- `late_close`: 7회 이후이고 점수 차 2점 이내 (세이버메트릭스의 Late & Close를 단순화)
- `is_home_batting`: 홈팀이 공격 중 (말 공격)

In [12]:
clean["score_diff"] = (clean["bat_score"] - clean["fld_score"]).astype("int16")
clean["risp"] = (clean["on_2b"].notna() | clean["on_3b"].notna()).astype("int8")
clean["late_close"] = ((clean["inning"] >= 7) & (clean["score_diff"].abs() <= 2)).astype("int8")
clean["is_home_batting"] = (clean["inning_topbot"] == "Bot").astype("int8")

clean = clean.merge(starters, on=["game_pk", "pitcher"], how="left")
clean["is_starter"] = clean["is_starter"].fillna(0).astype("int8")

clean[["is_strike", "is_borderline", "risp", "late_close", "is_home_batting", "is_starter"]].mean().round(3)

is_strike          0.315
is_borderline      0.163
risp               0.230
late_close         0.164
is_home_batting    0.490
is_starter         0.583
dtype: float64

- 판정구의 **31.5%가 스트라이크**다.
- **경계 공은 16.3%**, 약 12만 개다. 전체 규모로는 4.6절의 셀당 200개, 3.2절의 그룹당 2,000개 기준을 넉넉히 넘는다. 조건별 셀 크기는 `01_baseline`에서 교차표로 확인한다.
- 득점권 상황이 23.0%, 후반 접전이 16.4%다.
- 판정구의 58.3%를 그 경기 선발 투수가 던졌다.

## 5. 포수 프레이밍 점수

**프레이밍**은 포수가 공을 받는 동작으로 경계 공을 스트라이크처럼 보이게
하는 기술이다. 포수에 따라 경계 공 판정이 크게 달라지므로 통제 변수로 쓴다.

[Baseball Savant Catcher Framing 리더보드](https://baseballsavant.mlb.com/leaderboard/catcher-framing)에서
시즌별 CSV를 받아 `data/framing_YYYY.csv`로 저장한다 (용량이 작아 커밋한다).

- **점수 정의.** 리더보드의 `rv_tot`는 프레이밍으로 얻은 시즌 누적 득점이라
  출장이 많을수록 커진다. 포수의 실력만 보려면 비율이 맞으므로
  **판정구 1,000개당 프레이밍 득점**(`rv_tot / pitches × 1000`)을
  `framing_score`로 쓴다. 0이 리그 평균이다.
- **조인 키**는 MLB 선수 ID(`fielder_2` ↔ `id`)이고, **같은 시즌끼리만** 붙인다.
- 리더보드는 규정 출장 포수만 싣는다. 빠진 백업 포수는 **리그 평균(0)으로
  채우고, 몇 %가 채워졌는지 기록한다.**

In [13]:
def load_framing(year):
    path = DATA_DIR / f"framing_{year}.csv"
    if not path.exists():
        req = urllib.request.Request(FRAMING_URL.format(year=year), headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp:
            path.write_bytes(resp.read())
    return pd.read_csv(path, encoding="utf-8-sig").assign(game_year=year)


framing = pd.concat([load_framing(y) for y in SEASONS], ignore_index=True)
framing["framing_score"] = framing["rv_tot"] / framing["pitches"] * 1000

framing.groupby("game_year")["framing_score"].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
game_year,,,,,,,,
2024,58.0,-0.01,1.10,-2.10,-0.88,0.03,0.72,2.76
2025,57.0,-0.06,1.02,-2.59,-0.73,-0.09,0.49,2.81


리더보드에 시즌당 57~58명이 있다. `framing_score`는 약 -2.6 ~ +2.8 범위로, 판정구 1,000개당 최대 3점 안팎의 차이가 난다. 평균이 0 근처라 빠진 포수를 0(리그 평균)으로 채우는 것이 자연스럽다.

In [14]:
framing_key = framing[["game_year", "id", "framing_score"]].rename(columns={"id": "fielder_2"})
for col in ["game_year", "fielder_2"]:
    clean[col] = clean[col].astype("int64")
    framing_key[col] = framing_key[col].astype("int64")

clean = clean.merge(framing_key, on=["game_year", "fielder_2"], how="left")

framing_missing = clean.groupby("game_year")["framing_score"].apply(lambda s: s.isna().mean())
print("프레이밍 점수가 없어 0으로 채운 판정구 비율")
print((framing_missing * 100).round(1).astype(str) + "%")

clean["framing_score"] = clean["framing_score"].fillna(0.0)

프레이밍 점수가 없어 0으로 채운 판정구 비율
game_year
2024    10.8%
2025    12.7%
Name: framing_score, dtype: object


리더보드에 없는 백업 포수가 받은 판정구 **2024년 10.8%, 2025년 12.7%를 0으로 채웠다.** 이 공들은 포수 효과를 평균으로 가정한 셈이라 프레이밍 통제가 그만큼 약해진다. 결론에서 한계로 적는다.

리더보드의 `pitches`가 무엇을 세는지 확인한다. 점수를 `pitches`로 나눠
"판정구 1,000개당"이라고 부르려면, 이 값이 포수가 받은 판정구 수여야 한다.

In [15]:
ours = clean.groupby(["game_year", "fielder_2"]).size().rename("우리 판정구 수")
board = framing.set_index(["game_year", "id"]).rename_axis(["game_year", "fielder_2"])
compare = board[["name", "pitches", "rv_tot", "pct_tot"]].join(ours)

print(f"pitches / 우리 판정구 수 (중앙값): {(compare['pitches'] / compare['우리 판정구 수']).median():.3f}")
print(f"리더보드 최소 pitches: {compare['pitches'].min():,}")
compare.groupby("game_year").agg(포수=("name", "size"), 섀도존_스트라이크율=("pct_tot", "mean")).round(3)

pitches / 우리 판정구 수 (중앙값): 1.000
리더보드 최소 pitches: 2,549


,포수,섀도존_스트라이크율
game_year,,
2024,58,0.465
2025,57,0.420


`pitches`는 우리가 센 포수별 판정구 수와 **정확히 일치한다** (비율 1.000). 따라서 `framing_score`는 말 그대로 판정구 1,000개당 프레이밍 득점이다. 규정 출장 기준은 판정구 약 2,550개다.

리더보드의 `pct_tot`(존 경계 부근 '섀도 존'의 스트라이크 비율)도 2024년 46.5%에서 2025년 42.0%로 떨어졌다. Savant 자체 집계에서도 아래에서 볼 2025년의 존 축소가 같은 방향으로 나타난다.

## 6. 저장

컬럼을 읽기 좋은 순서로 정리하고 타입을 줄인다. 선수·경기 ID는 정수
(`int32`), 0/1 변수는 `int8`이다. 주자 컬럼(`on_1b` 등)은 주자의 선수 ID이고
빈 베이스는 결측이라 결측 가능한 정수(`Int32`)로 둔다.

In [16]:
ID_COLS = ["game_pk", "game_year", "at_bat_number", "pitch_number", "pitcher", "batter", "fielder_2",
           "balls", "strikes", "outs_when_up", "inning", "bat_score", "fld_score"]
for col in ID_COLS:
    clean[col] = clean[col].astype("int32")
for col in ["on_1b", "on_2b", "on_3b"]:
    clean[col] = clean[col].astype("Int32")

ORDER = [
    # 식별
    "game_date", "game_pk", "game_year", "game_type", "at_bat_number", "pitch_number",
    # 판정
    "description", "is_strike",
    # 위치
    "plate_x", "plate_z", "sz_top", "sz_bot", "plate_z_norm", "zone_dist", "is_borderline",
    # 상황
    "balls", "strikes", "outs_when_up", "inning", "inning_topbot",
    "on_1b", "on_2b", "on_3b", "bat_score", "fld_score",
    "score_diff", "risp", "late_close", "is_home_batting",
    # 선수
    "pitcher", "batter", "fielder_2", "stand", "p_throws", "is_starter",
    "pitch_type", "release_speed", "home_team", "away_team",
    # 통제
    "framing_score",
]
assert sorted(ORDER) == sorted(clean.columns), set(ORDER) ^ set(clean.columns)
clean = clean[ORDER].sort_values(["game_date", "game_pk", "at_bat_number", "pitch_number"], ignore_index=True)

out_path = DATA_DIR / "clean.parquet"
clean.to_parquet(out_path, index=False)
print(f"{out_path}: {len(clean):,}행 × {clean.shape[1]}열, {out_path.stat().st_size / 1e6:.1f} MB")

../data/clean.parquet: 735,718행 × 40열, 43.9 MB


735,718행 × 40열, **43.9MB**다. GitHub 100MB 제한 안이라 그대로 커밋한다 (가이드라인 5.4절).

마지막으로 저장한 파일을 다시 읽어, 정의대로 만들어졌는지 확인한다.

In [17]:
check = pd.read_parquet(out_path)

assert check["description"].isin(CALLED).all()
assert check[["plate_x", "plate_z", "sz_top", "sz_bot", "framing_score"]].notna().all().all()
assert not check.duplicated(["game_pk", "at_bat_number", "pitch_number"]).any()
assert (check["is_strike"] == (check["description"] == "called_strike")).all()

summary = check.groupby("game_year").agg(
    판정구=("is_strike", "size"),
    스트라이크_비율=("is_strike", "mean"),
    경계공=("is_borderline", "sum"),
    경계공_스트라이크_비율=("is_strike", lambda s: s[check.loc[s.index, "is_borderline"] == 1].mean()),
)
summary.round(3)

,판정구,스트라이크_비율,경계공,경계공_스트라이크_비율
game_year,,,,
2024,366873,0.317,59505,0.502
2025,368845,0.313,60548,0.427


검증은 모두 통과했다. 그런데 **경계 공 스트라이크 비율이 2024년 50.2%, 2025년 42.7%로 7.5%p 차이가 난다.** 전체 스트라이크 비율은 거의 같다 (31.7% vs 31.3%). 아래에서 이 차이를 들여다본다.

### 시즌 간 차이 확인

경계 공 스트라이크 비율이 두 시즌 사이에 다르다. 두 시즌을 합쳐 쓰는 것이
전제이므로 (가이드라인 1.1절), 차이가 언제 생겼고 어디서 나는지 확인한다.
먼저 월별 추이다.

In [18]:
borderline = check[check["is_borderline"] == 1]
borderline.pivot_table(
    index=borderline["game_date"].dt.month.rename("월"),
    columns="game_year", values="is_strike", aggfunc="mean",
).round(3)

game_year,2024,2025
월,,
3,0.495,0.483
4,0.499,0.425
5,0.492,0.435
6,0.500,0.418
7,0.504,0.420
8,0.514,0.423
9,0.502,0.428


2024년은 시즌 내내 50% 안팎으로 일정하다. 2025년은 3월(48.3%)까지 비슷하다가 **4월부터 42~43%로 내려가 시즌 끝까지 유지된다.** 시즌 중에 서서히 변한 것이 아니라 시즌 초에 한 번 바뀐 모습이다.

차이가 존의 어느 쪽에서 나는지 본다. 경계 공을 가장 가까운 경계가 좌우인지,
상단인지, 하단인지로 나눈다. 좌우 경계는 존 높이 측정값(`sz_top`, `sz_bot`)과
무관하므로, 여기서도 차이가 나면 높이 측정 문제만으로는 설명되지 않는다.

In [19]:
dx = (PLATE_HALF + BALL_RADIUS) - borderline["plate_x"].abs()
dz = np.minimum(
    (borderline["sz_top"] + BALL_RADIUS) - borderline["plate_z"],
    borderline["plate_z"] - (borderline["sz_bot"] - BALL_RADIUS),
)
mid_height = (borderline["sz_top"] + borderline["sz_bot"]) / 2
edge = np.where(dx < dz, "좌우", np.where(borderline["plate_z"] > mid_height, "상단", "하단"))

borderline.pivot_table(
    index=pd.Series(edge, index=borderline.index, name="가까운 경계"),
    columns="game_year", values="is_strike", aggfunc=["mean", "size"],
).round(3)

mean          size       
game_year   2024   2025   2024   2025
가까운 경계                               
상단         0.369  0.279  10181  10583
좌우         0.585  0.497  34321  34885
하단         0.402  0.369  15003  15080

상단·하단·좌우 **세 경계 모두에서** 2025년이 낮다. 좌우 경계(58.5% → 49.7%)는 존 높이 측정값과 무관하므로, 존 높이 측정 방식이 바뀐 탓만은 아니다.

In [20]:
bins = np.arange(-0.3, 0.31, 0.1)
check.pivot_table(
    index=pd.cut(check["zone_dist"], bins).rename("zone_dist 구간 (ft)"),
    columns="game_year", values="is_strike", aggfunc="mean", observed=True,
).round(3)

game_year,2024,2025
zone_dist 구간 (ft),,
"(-0.3, -0.2]",0.053,0.031
"(-0.2, -0.1]",0.163,0.116
"(-0.1, 2.78e-17]",0.347,0.272
"(2.78e-17, 0.1]",0.670,0.594
"(0.1, 0.2]",0.857,0.830
"(0.2, 0.3]",0.937,0.929


존 바깥(음수 구간)과 존 안쪽 가장자리(0~0.1ft) 모두에서 2025년 스트라이크 비율이 낮다. 경계에서 멀어질수록 두 시즌이 같아진다. **2025년의 판정 존이 전체적으로 조금 안쪽으로 좁아진 것**이다. 앞의 프레이밍 리더보드(`pct_tot`)에서도 같은 방향이 보였다.

**분석에 주는 의미.** 두 시즌을 그대로 합치면 이 시즌 차이가 상황·선수 효과에 섞여 들어갈 수 있다. 시즌별 존 위치는 좌우 타자 차이처럼 "존이 어디인가"의 일부이지 우리가 재려는 편향이 아니다. 그래서 다음 두 가지를 한다 (가이드라인 1.1, 2.5, 4.8절).

1. **모든 모델에 `game_year`를 넣어** 시즌 간 평균 차이를 흡수한다.
2. 핵심 결과를 **시즌별로 나눠 다시 구한다.** 효과가 비슷하면 존이 바뀌어도 편향은 남았다는 뜻이고, 2025년에 줄었으면 존이 좁아진 시즌에 편향도 줄었다는 발견이 된다.

## 정리

- 최종 데이터: `data/clean.parquet`
- 원본: `data/raw/statcast_YYYY_MM.parquet` (커밋하지 않음, 이 노트북으로 재생성)
- 프레이밍: `data/framing_2024.csv`, `data/framing_2025.csv`

기준 모델 예측값 `baseline_prob`은 다음 노트북 `01_baseline.ipynb`에서
이 파일에 컬럼으로 덧붙인다.